In [3]:
import sys
import subprocess

# Force pip to install directly into the specific Python interpreter running this notebook
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'beautifulsoup4', 'requests', 'scikit-learn'])
print("✅ Packages successfully installed into the current kernel!")

✅ Packages successfully installed into the current kernel!


In [10]:
import os
import requests
import time
import re

TARGET_CATEGORIES = {
    "Lotus_Mahal": "Lotus_Mahal_(Hampi)",
    "Elephant_Stables": "Elephant_stables_at_Hampi",
    "Hampi_Bazaar": "Hampi_Bazaar",
    "Hemakuta_temple_hill_complex": "Hemakuta_temple_hill_complex",
    "Monolithic_Bull": "Monolithic_Bull_(Hampi)",
    "Royal_Centre": "Royal_Centre_(Hampi)"
}

SAVE_DIR = "../data/train_images"
os.makedirs(SAVE_DIR, exist_ok=True)
API_URL = "https://commons.wikimedia.org/w/api.php"

headers = {
    "User-Agent": "HampiClassifierBot/2.0 (Python/Requests) - Testing Linear Probe"
}
session = requests.Session()
session.headers.update(headers)

def sanitize_filename(filename):
    """Removes invalid characters for Windows file paths."""
    return re.sub(r'[\\/*?:"<>|]', "_", filename)

for folder_name, category_name in TARGET_CATEGORIES.items():
    cat_dir = os.path.join(SAVE_DIR, folder_name)
    os.makedirs(cat_dir, exist_ok=True)
    
    print(f"\nFetching images for category: {category_name}")
    
    params = {
        "action": "query",
        "format": "json",
        "list": "categorymembers",
        "cmtitle": f"Category:{category_name}",
        "cmtype": "file",
        "cmlimit": "40"
    }
    
    try:
        response = session.get(API_URL, params=params, timeout=10).json()
        members = response.get('query', {}).get('categorymembers', [])
        
        if not members:
            # Fallback to search
            search_params = {
                "action": "query",
                "format": "json",
                "list": "search",
                "srsearch": f"{category_name.replace('_', ' ')} filetype:bitmap",
                "srlimit": "40"
            }
            search_resp = session.get(API_URL, params=search_params, timeout=10).json()
            members = search_resp.get('query', {}).get('search', [])
            
        downloaded = 0
        for member in members:
            if downloaded >= 20:
                break
                
            title = member.get('title', '')
            if not title.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            # Request an 800px thumbnail instead of the original file
            img_params = {
                "action": "query",
                "format": "json",
                "titles": title,
                "prop": "imageinfo",
                "iiprop": "url",
                "iiurlwidth": "800" 
            }
            
            img_resp = session.get(API_URL, params=img_params, timeout=10).json()
            pages = img_resp.get('query', {}).get('pages', {})
            
            for page_id, page_data in pages.items():
                image_info = page_data.get('imageinfo', [{}])[0]
                # Try getting the thumburl; if it fails, fallback to url
                image_url = image_info.get('thumburl') or image_info.get('url')
                
                if image_url:
                    clean_title = sanitize_filename(title.replace('File:', '').replace(' ', '_'))
                    save_path = os.path.join(cat_dir, clean_title)
                    
                    if not os.path.exists(save_path):
                        # Add a small delay to avoid 429 Too Many Requests
                        time.sleep(0.5)
                        
                        dl_resp = session.get(image_url, timeout=15)
                        
                        # Verify we actually got an image before saving
                        if dl_resp.status_code == 200 and 'image' in dl_resp.headers.get('Content-Type', ''):
                            with open(save_path, 'wb') as f:
                                f.write(dl_resp.content)
                            
                            downloaded += 1
                            if downloaded % 5 == 0:
                                print(f"  ...downloaded {downloaded} valid images")
                        else:
                            print(f"  ...Skipping {clean_title}: Not an image or blocked by server.")
                    else:
                        downloaded += 1

        print(f"✅ Finished: {downloaded} images for {folder_name}")
        
    except Exception as e:
        print(f"❌ Error fetching {category_name}: {e}")

print("\n🎉 All scraping complete! You can now run the Training cell.")


Fetching images for category: Lotus_Mahal_(Hampi)
  ...Skipping Flat_elevation_of_Lotus_Mahal,_Hampi.jpg: Not an image or blocked by server.
  ...Skipping Front_view_of_Lotus_Mahal.jpg: Not an image or blocked by server.
  ...Skipping Front_view..jpg: Not an image or blocked by server.
  ...Skipping Grandeur_of_Hampi-1.jpg: Not an image or blocked by server.
  ...Skipping Grate_Lotus_Palace.jpg: Not an image or blocked by server.
  ...Skipping Hampi_(6160661479).jpg: Not an image or blocked by server.
  ...Skipping Hampi_(6160662895).jpg: Not an image or blocked by server.
  ...Skipping Hampi_-_Lotus_Mahal.jpg: Not an image or blocked by server.
  ...Skipping Hampi_0028.jpg: Not an image or blocked by server.
  ...downloaded 5 valid images
  ...Skipping Hampi_architecture_0129.jpg: Not an image or blocked by server.
  ...Skipping Hampi_aug09_226.jpg: Not an image or blocked by server.
  ...Skipping Hampi_DSC_0357.jpg: Not an image or blocked by server.
  ...Skipping Hampi_DSC_0358.jpg

In [11]:
import sys
import pickle
import numpy as np
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Ensure we can import from the model folder
sys.path.insert(0, os.path.abspath('..'))
from model.clip_model import HampiCLIPModel

# 1. Load CLIP Model merely as a feature extractor
print("Loading CLIP for feature extraction...")
clip_model = HampiCLIPModel()
clip_model.load()

# 2. Extract Features from the scraped images
TRAIN_DIR = '../data/train_images' 

X_features = []
y_labels = []

print(f"Extracting features from {TRAIN_DIR}...")
for folder_name in os.listdir(TRAIN_DIR):
    folder = os.path.join(TRAIN_DIR, folder_name)
    if not os.path.isdir(folder): continue
        
    for fname in os.listdir(folder):
        if not fname.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')): continue
            
        try:
            img = Image.open(os.path.join(folder, fname)).convert('RGB')
            # Extract 768-dim vector
            feat = clip_model._encode_image(img).cpu().numpy().flatten()
            X_features.append(feat)
            y_labels.append(folder_name)
        except Exception as e:
            pass

X_features = np.array(X_features)
y_labels = np.array(y_labels)
print(f"Extracted {len(X_features)} total images.")

# 3. Train Logistic Regression
X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)

print("\nTraining Linear Probe (Logistic Regression)...")
clf = LogisticRegression(max_iter=1000, C=1.0)
clf.fit(X_train, y_train)

preds = clf.predict(X_test)
acc = accuracy_score(y_test, preds)
print(f"Internal Validation Accuracy: {acc*100:.1f}%")

# 4. Save the trained classifier exactly where clip_model.py can find it
model_path = '../model/hampi_classifier.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(clf, f)
print(f"✅ Saved linear probe to {model_path}")

Loading CLIP for feature extraction...
⚠️ No classifier.pkl found! Falling back to Zero-Shot Text Prompts.
✅ CLIP loaded on cpu
Extracting features from ../data/train_images...
Extracted 49 total images.

Training Linear Probe (Logistic Regression)...
Internal Validation Accuracy: 50.0%
✅ Saved linear probe to ../model/hampi_classifier.pkl
